# 04 — Retrieve graph content with Apache AGE
This is not a mock graph. It queries the real `course_graph` projection created from PostgreSQL course/module/source/technology data.

In [ ]:
from course_knowledge.database import connect
from course_knowledge.settings import database_settings
with connect(database_settings()) as db, db.cursor() as cur:
    cur.execute('SET search_path = ag_catalog, "$user", public')
    cur.execute("SELECT * FROM cypher('course_graph', $$ MATCH (course:Course)-[:HAS_MODULE]->(module:Module)-[:HAS_SOURCE]->(source:Source) RETURN course,module,source LIMIT 10 $$) AS (course agtype,module agtype,source agtype)")
    for row in cur.fetchall(): print(row)

In [ ]:
from course_knowledge.graph_retrieval import GraphExpander
from course_knowledge.hybrid import HybridRetriever
from course_knowledge.embeddings import AzureOpenAIEmbedder
from course_knowledge.settings import azure_openai_settings
question = 'How is MCP connected to GitHub Copilot?'
with connect(database_settings()) as db:
    hits = HybridRetriever(db, AzureOpenAIEmbedder.from_settings(azure_openai_settings())).search(question, 3)
    expander = GraphExpander(db)
    for hit in hits:
        print(hit.candidate.metadata['source'])
        print(expander.from_source(hit.candidate.metadata['source_id']))

## Exercise
Explain the difference between retrieved evidence chunks and graph context. Graph edges provide verified curriculum structure; source chunks provide factual teaching evidence.